# Chapter 2: The Model Validation Framework

**The question.** A money-laundering model can have a small average error and still make large errors on one group of customers. How do we find such a group, show that the weakness is real and report it so that every number can be traced back to its source? This notebook follows one test through that whole process. Chapters 3 to 10 then take each stage apart.

**The steps.**

1. Define one test: a rule that picks a group of customers, scored on 40,000 *discovery* customers.
2. Compare the 400 tests a search produced and pick one by a fixed rule.
3. Freeze the picked test and check it on 200,000 new customers.
4. Check that a reported number matches the stored evidence.

**Result.** The picked group holds 11.6% of the discovery customers, where the model's error is 0.062 higher than for everyone else. On new customers the gap is 0.058, a little smaller but clearly real. The evidence check accepts the reported number, rejects a wrong one and flags a claim that has no evidence.

**Map of this notebook.**

| Step | Question | Where to look |
|---|---|---|
| 1 | What is one test, and how is it scored? | second table, `contrast` and `share` |
| 2 | Which of the 400 tests is picked? | the frontier table and the `Selected candidate` line |
| 3 | Does the weakness hold on new customers? | the discovery-versus-confirmation table |
| 4 | Can a reported number be traced to its source? | the `verdict` column |

**Terms.**

- *Brier loss*: the model's error on one customer, the squared gap between the predicted probability and what happened (0 or 1).
- *Region*: the group of customers a test picks. Its *share* is the fraction of customers in it.
- *Contrast*: the region's average Brier loss minus that of everyone else. Higher means a weaker spot.
- *Medoid*: a real customer used as a cluster center. A test uses 4 medoids to split customers into 4 clusters; the code calls clusters *cells*.
- *Inner Array*: the list of candidate tests. *Outer Array*: the conditions each test is scored under; here there is one, the discovery sample (Sections 2.4 and 2.5).
- *Frontier*: the tests for which no other test has both a larger contrast and a larger share.

**Data.** The records are in `book/evidence/aml_medoid_loop_leaf_output_20260921/`. The customers are simulated. The model and the error model were fitted once and are only read here; no model is fitted and no language model is called. Chapter 3 uses a different way of describing customers and a different confirmation sample, so its numbers should not be mixed with these.

Run the cells in order. The `assert` lines check the recomputed numbers against the saved results.

## Setup and source checks

Loads the saved results and checks, by SHA-256 fingerprint, that the files match the recorded run. Select the kernel of the environment where `aiv` is installed.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Step 1: One test and its score

A test is a rule that picks a group of customers before any error is measured. Here the rule has two parts. Four medoids split the customers into 4 clusters, each customer joining the nearest medoid. Then the eligible cluster (at least 50 customers and a share between 1% and 80%) with the largest contrast is picked:

$$\Delta = \frac{1}{n_m}\sum_{i:\,m_i} L_i \;-\; \frac{1}{n-n_m}\sum_{i:\,\neg m_i} L_i, \qquad L_i=(y_i-p_i)^2 .$$

"Nearest" is measured in the error model's view of a customer: the 120 amounts its trees add to the predicted error. The cell shows the first test in the search history for seed 73101.

In [2]:
run = read(RUN / "twinning_73101.json")
first = run["records"][0]
display(pd.DataFrame([{"medoids": first["medoids"], "selected_cells": first["region"]["cells"],
                       "outer_array": protocol["outer_array"][0]}]))
display(pd.DataFrame([{"cell_sizes": first["counts"], "region_n": first["region"]["n"],
                       "share": first["region"]["share"], "region_brier": first["region"]["loss"],
                       "rest_brier": first["region"]["rest_loss"], "contrast": first["region"]["contrast"]}]))
print("Predictor:", protocol["base_model"])
print("Auxiliary error model:", protocol["auxiliary"])
print("Eligible region share:", protocol["region_min_share"], "to", protocol["region_max_share"],
      "with at least", protocol["region_min_n"], "observations")
print("Search objectives:", protocol["objectives"])

,medoids,selected_cells,outer_array
0,"[95, 984, 2678, 2975]",[1],current rare-event population; same fixed 4000...


,cell_sizes,region_n,share,region_brier,rest_brier,contrast
0,"[12978, 8214, 6515, 12293]",8214,0.205350,0.054179,0.012384,0.041795


Predictor: unchanged saved logistic model
Auxiliary error model: unchanged saved depth-two 120-tree model
Eligible region share: 0.01 to 0.8 with at least 50 observations
Search objectives: ['largest eligible region Brier contrast versus pooled rest', 'selected region population share']


**Read the output.** **In one line: this test picks cluster 1, 20.5% of customers, where the model's error is 0.042 higher than for everyone else.**

**Columns, first table.** `medoids`: the 4 centers, as positions in a fixed pool of 3,000 customers. `selected_cells`: the cluster the rule picked. `outer_array`: the one condition every test is scored under, the same 40,000 discovery customers.

**Columns, second table.** `cell_sizes`: customers in each of the 4 clusters (they add up to 40,000). `region_n` and `share`: size of the picked cluster, in customers and as a fraction. `region_brier` and `rest_brier`: the model's average error inside and outside. `contrast`: the difference, 0.054179 − 0.012384 = 0.041795.

**Notice.** The medoids and the rule are the test's inputs; the contrast and share are its outputs. Those two outputs are what the search tries to make large.

## Step 2: Compare the tests and pick one

The search produced 400 tests. Larger contrast usually comes with a smaller group, so there is no single best test. The frontier keeps every test that no other test beats on both contrast and share (Section 2.6). One test must then be picked for checking, by a rule fixed before any new data existed: the largest contrast among frontier tests with a share of at least 10%.

In [3]:
kept = frontier(run["records"])
assert {r["id"] for r in kept} == set(run["frontier_ids"])
selected = max((r for r in kept if r["objectives"][1] >= .10),
               key=lambda r: (r["objectives"][0], r["objectives"][1], -r["id"]))
assert selected == run["primary"]
print("Candidates evaluated:", len(run["records"]), "Frontier size:", len(kept))
print("Primary policy:", protocol["primary_policy"])
display(pd.DataFrame([dict(candidate=r["id"], contrast=r["objectives"][0], share=r["objectives"][1],
                           eligible=r["objectives"][1] >= .10) for r in kept]))
print("Selected candidate:", selected["id"], "cells", selected["region"]["cells"],
      "contrast", round(selected["region"]["contrast"], 6), "share", selected["region"]["share"])

Candidates evaluated: 400 Frontier size: 39
Primary policy: maximum discovery contrast on Pareto frontier with share >= 0.10; tie larger share then earlier ID


,candidate,contrast,share,eligible
0,384,0.238311,0.010425,False
1,378,0.230937,0.010675,False
2,308,0.200790,0.010825,False
3,213,0.200212,0.010900,False
4,365,0.200071,0.010975,False
5,244,0.198873,0.011075,False
6,323,0.197944,0.011125,False
7,306,0.187947,0.011150,False
8,291,0.183409,0.012525,False
9,320,0.181077,0.014625,False


Selected candidate: 279 cells [0] contrast 0.061616 share 0.115625


**Read the output.** **In one line: 39 of the 400 tests make the frontier, and candidate 279 is picked: cluster 0, 11.6% of customers, contrast 0.062.**

**Columns.** `candidate`: the test's number in the search history. `contrast` and `share`: its two scores. `eligible`: whether its share is at least 10%, the minimum for being picked.

**Notice.** Contrast falls as share rises: candidate 384 reaches 0.238 with 1.04% of customers, while candidate 96 covers 62.3% with 0.020. The first eligible row, candidate 279, is picked. Because it was picked for having the largest contrast, its score is likely too high; Chapter 9 measures this selection effect.

## Step 3: Freeze the test and check it on new customers

Before checking, the whole rule is frozen: the model, the error model, the 4 medoids and the picked cluster. A new customer can then be placed in or out of the group from its features alone, without looking at its outcome (Section 2.8). The cell replays that placement for the first 5,000 of 200,000 new customers and compares it with the saved placement. It then recomputes the contrast on all 200,000 with a range of plausible values. The ranges are widened (Bonferroni) because the study makes 19 claims on the same data: 13 confirmation contrasts and six paired differences.

In [4]:
key = "twinning_73101"
choice = next(r for r in frozen["finalists"] if r["key"] == key)
assert choice["discovery"] == selected
labels, _ = assignments(choice)
print("Frozen cells:", choice["discovery"]["region"]["cells"],
      "Replayed rows:", len(labels), "Region members among them:", int(np.isin(labels, choice["discovery"]["region"]["cells"]).sum()))
from scipy.stats import norm
z = norm.ppf(1 - .05 / (2 * results["family_size"]))
assert np.isclose(z, results["critical_value"])
loss, masks = confirmation()
recomputed, _ = measure(loss, masks[key], z)
saved = results["confirmation"][key]
for field in ["n", "share", "brier", "rest_brier", "contrast", "standard_error"]:
    assert np.isclose(recomputed[field], saved[field], rtol=0, atol=1e-12), field
assert np.allclose(recomputed["simultaneous_ci"], saved["simultaneous_ci"], rtol=0, atol=1e-12)
print("Critical value:", z, "for", results["family_size"], "claims")
display(pd.DataFrame([{"split": "discovery", "share": selected["region"]["share"], "contrast": selected["region"]["contrast"]},
                      {"split": "confirmation", "share": saved["share"], "contrast": saved["contrast"]}]).set_index("split"))
display(pd.DataFrame([saved], index=[key]))

Frozen cells: [0] Replayed rows: 5000 Region members among them: 560
Critical value: 3.0077865564732638 for 19 claims


,share,contrast
split,,
discovery,0.115625,0.061616
confirmation,0.113925,0.057940


,n,share,brier,rest_brier,contrast,standard_error,simultaneous_ci
twinning_73101,22785,0.113925,0.072584,0.014645,0.057940,0.001525,"[0.05335135577493813, 0.06252789293682992]"


**Read the output.** **In one line: on new customers the weakness holds, with a contrast of 0.058 and a plausible range of 0.053 to 0.063.**

**First printed lines.** The replayed placement matches the saved one; 560 of the 5,000 replayed customers fall in the group. The critical value 3.0078 is the widening factor for 19 claims.

**Columns, first table.** `share` and `contrast` of the picked group on the discovery customers and on the new (confirmation) customers.

**Columns, second table.** The full measurement on new customers. `n`: 22,785 customers in the group. `brier` and `rest_brier`: average error inside (0.072584) and outside (0.014645). `standard_error`: the uncertainty of the contrast. `simultaneous_ci`: the plausible range, [0.053351, 0.062528].

**Notice.** The range is far from zero, so the weak spot is real in this simulated population. The contrast dropped from 0.061616 to 0.057940, the optimism noted in Step 2. This measures where the model errs; it does not show that the model can be fixed there.

## Step 4: Check a reported number against the evidence

A report restates numbers, and each should be traceable to where it came from. The evidence store holds one record per measured number, with an address (`id`), the value, the source file and where in that file the value sits. `verify` checks a claimed number against its record and returns a verdict: *Supported* if it matches (within 0.5×10⁻⁶), *Contradicted* if the record holds a different value, *Absent* if no record exists and *Unscorable* if the claim has the wrong type. The cell checks three claims.

In [5]:
from book.parallel_evidence.build import OUT, verify
store = read(OUT / "store.json")
address = f"region:confirmation/{key}/contrast"
value = round(results["confirmation"][key]["contrast"], 6)
claims = [("reported contrast", address, value),
          ("altered contrast", address, 0.10),
          ("predictor repair", "region:experiment/predictor_repair_gain", 0.01)]
rows = [dict(claim=name, id=id_, value=claimed, verdict=verify(store, {"id": id_, "value": claimed})["verdict"])
        for name, id_, claimed in claims]
assert [r["verdict"] for r in rows] == ["Supported", "Contradicted", "Absent"]
display(pd.DataFrame(rows))
display(pd.DataFrame([verify(store, {"id": address, "value": value})["evidence"]]))

,claim,id,value,verdict
0,reported contrast,region:confirmation/twinning_73101/contrast,0.057940,Supported
1,altered contrast,region:confirmation/twinning_73101/contrast,0.100000,Contradicted
2,predictor repair,region:experiment/predictor_repair_gain,0.010000,Absent


,value,source,source_sha256,pointer
0,0.057940,book/evidence/aml_medoid_loop_leaf_output_2026...,fad0af92d04996ccb01f907412dc018c401f39d05443c6...,/confirmation/twinning_73101/contrast


**Read the output.** **In one line: the correct contrast is Supported, a wrong value is Contradicted and an unmeasured claim is Absent.**

**Columns, first table.** `claim`: what is being claimed. `id`: its address in the evidence store. `value`: the claimed number. `verdict`: the result.

**Columns, second table.** The record behind the first claim. `value`: the stored number. `source`: the results file it came from. `source_sha256`: that file's fingerprint, so a changed file is detected. `pointer`: where the value sits inside the file.

**Notice.** "Predictor repair" is Absent, not Contradicted, because the study never measured it: there is no record to disagree with. The verifier only compares an address and a number. Pulling claims out of written text, and judging whether a sentence interprets a number correctly, are separate problems treated in Chapters 7 and 10.

## What we learned

One test went through the whole process. A rule defined a group of customers, the discovery data scored it, the frontier kept the tradeoffs among 400 tests and a fixed rule picked one. The frozen test was then checked on new customers, where the weakness held at a slightly smaller size, and the reported number was traced back to its source record.

Each stage answers its own question: discovery scores guide the choice, new customers test it and the verifier checks what is reported. Chapter 3 looks at how the error model describes customers, and Chapter 4 at the search that produced the 400 tests.


## What to try

1. **What the search changed.** Compare the first test with candidate 279: cluster sizes, share and contrast. Use `operation` and `parent_id` to trace how candidate 279 was made.
2. **The 10% minimum.** Pick from `kept` with minimums of 5% and 20% instead. Which candidates win? Why must the minimum be fixed before looking at new customers?
3. **Other seeds.** Load `twinning_73102.json` and `twinning_73103.json`, recompute their frontiers and picks and compare them with their frozen finalists. The three seeds share one discovery sample, so agreement is not an independent replication.
4. **Verifier verdicts.** Call `verify` with the discovery contrast `selected["region"]["contrast"]` at the confirmation address, and with the string `"0.057940"`. Why is the first Contradicted and the second Unscorable?
